# Smart Agricultural Price Prediction System
## Notebook 02: Feature Engineering

### Objective
In this notebook, we transform our clean time-series mandi price dataset into an ML-ready dataset by building meaningful features. We will construct calendar features, seasonality maps, autoregressive lags, moving averages, growth ratios, and volatility indicators. 

To prevent data leakage, all time-series features are calculated *chronologically* and grouped specifically by **Crop, State, District, and Market**. Lags and rolling features are shifted appropriately to ensure we do not predict the price of today using information from today.

In [ ]:
import os
import pandas as pd
import numpy as np

# Set display options
pd.set_option("display.max_columns", None)

print("[+] Libraries imported.")

--- 
## 1. Load Clean Master Dataset

In [ ]:
# Load cleaned data
df = pd.read_csv("../data/processed/cleaned_master.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")
print(f"Loaded dataset with shape: {df.shape}")
df.head(3)

--- 
## 2. Apply Feature Engineering
We call our modular function from `src/features.py` to extract all features.

In [ ]:
import sys
sys.path.append("..")
from src.features import build_features

# Apply the programmatic builder
df_featured = build_features(df)
print(f"Shape after feature generation: {df_featured.shape}")

# Display some sample rows with features populated
df_featured.tail(5)

--- 
## 3. Drop NaNs caused by Lags and Rolling Windows
Since calculating 7-day lags or rolling averages requires 7 prior historical entries, the first 7 entries for each market/crop combination will have missing values (`NaN`). We must drop these rows to feed a clean matrix to scikit-learn models.

In [ ]:
print("NaN counts before cleaning:")
print(df_featured.isnull().sum()[df_featured.isnull().sum() > 0])

# Drop NaNs
df_clean_featured = df_featured.dropna()
print(f"\nFinal dataset shape: {df_clean_featured.shape}")

# Save to processed folder
df_clean_featured.to_csv("../data/processed/featured_master.csv", index=False)
print("[+] Saved featured_master.csv successfully.")

--- 
## 4. Feature Summary & Explanation

Here is the logical breakdown of the engineered features and how they support our agricultural price forecasting models:

| Feature Category | Feature Name | Description | Rationale (Why it is useful) | Impact on ML Models |
| :--- | :--- | :--- | :--- | :--- |
| **Calendar** | `day` | Day of the month (1-31) | Captures monthly cyclical trends (e.g. traders balancing books at month-end). | Helps trees split on salary/trade schedules. |
| **Calendar** | `month` | Month of year (1-12) | Captures annual crop cycles and seasonal weather impacts. | Crucial for identifying harvest and lean periods. |
| **Calendar** | `year` | Year of record (e.g. 2020-2026) | Captures macroeconomics and long-term price inflation. | Helps linear trend models adjust for inflation. |
| **Calendar** | `week_number` | ISO week of year (1-53) | Captures high-frequency seasonal patterns better than month. | Fills gaps in seasonal weather/cropping bounds. |
| **Season** | `season` | Mapping of month to Indian seasons (Winter, Summer, Monsoon, Post-Monsoon) | Groups crop prices by agricultural seasons (Kharif, Rabi, Zaid harvests). | High importance split feature for tree algorithms. |
| **Autoregressive (Lags)** | `lag_1` | Price of the crop on the previous available trading day | In time-series, the most recent value is the strongest predictor of today's price. | Establishes the baseline value for regression models. |
| **Autoregressive (Lags)** | `lag_2` | Price of the crop two available trading days ago | Helps the model capture short-term velocity/direction of price changes. | Allows models to compute basic acceleration gradients. |
| **Autoregressive (Lags)** | `lag_7` | Price of the crop on the same day in the previous week | Captures weekly cyclical dependencies and market day effects. | Stabilizes predictions against day-of-week noise. |
| **Rolling Window** | `ma_3` | 3-day rolling average of modal prices (shifted to avoid leakage) | Smooths out high-frequency daily price fluctuations. | Helps models filter out transaction-level noise. |
| **Rolling Window** | `ma_7` | 7-day rolling average of modal prices (shifted to avoid leakage) | Captures the medium-term price trend direction. | Represents a robust baseline price trend. |
| **Price Growth** | `price_change_pct` | Day-over-day rate of change in price | Measures the speed of inflation or market panic. | Helps linear models predict price momentum. |
| **Volatility** | `price_volatility` | 7-day rolling standard deviation of price changes | Captures uncertainty or panic in the local mandi. | Informs models when to make cautious/less aggressive predictions. |